# Practica 2 - Ejercicio 4: Rendimiento en examenes

**Enunciado:** Utilizando `dataset_rendimiento.csv` (235 estudiantes), ajustar un modelo de
regresion que prediga el rendimiento promedio en examenes a partir de las horas de estudio
y el tipo de desayuno.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats
%matplotlib inline

In [2]:
df = pd.read_csv("../Datasets/Parte_2/dataset_rendimiento.csv")
print(df.head(10))
print(f"\nShape: {df.shape}")
print(f"\nCategorias de Desayuno: {df['Desayuno'].unique()}")
print(df["Desayuno"].value_counts())

   Horas_de_Estudio   Desayuno  Rendimiento
0                 9  Saludable         64.0
1                 9    Ninguno         84.2
2                12    Ninguno         66.9
3                10  Saludable         82.9
4                12    Ninguno         90.6
5                 9    Ninguno         81.5
6                 7  Saludable         95.0
7                11  Saludable         92.0
8                 4    Ninguno         84.0
9                 2    Ninguno         64.5

Shape: (235, 3)

Categorias de Desayuno: <StringArray>
['Saludable', 'Ninguno', 'Regular']
Length: 3, dtype: str
Desayuno
Ninguno      89
Regular      75
Saludable    71
Name: count, dtype: int64


## 1. Ajuste del modelo

In [3]:
# Desayuno es categorica: Ninguno (referencia), Regular, Saludable
mod = smf.ols("Rendimiento ~ Horas_de_Estudio + C(Desayuno)", data=df)
res = mod.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:            Rendimiento   R-squared:                       0.386
Model:                            OLS   Adj. R-squared:                  0.378
Method:                 Least Squares   F-statistic:                     48.47
Date:                Thu, 04 Jun 2026   Prob (F-statistic):           2.46e-24
Time:                        18:02:02   Log-Likelihood:                -880.27
No. Observations:                 235   AIC:                             1769.
Df Residuals:                     231   BIC:                             1782.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

## 2. Significancia de las variables

In [4]:
print(f"R2 = {res.rsquared:.4f}")
print("\nSignificancia de los coeficientes (alpha=0.05):")
for var, pv in res.pvalues.items():
    sig = "SIGNIFICATIVA" if pv < 0.05 else "No significativa"
    print(f"  {var}: p={pv:.4f} -> {sig}")

R2 = 0.3863

Significancia de los coeficientes (alpha=0.05):
  Intercept: p=0.0000 -> SIGNIFICATIVA
  C(Desayuno)[T.Regular]: p=0.5361 -> No significativa
  C(Desayuno)[T.Saludable]: p=0.0242 -> SIGNIFICATIVA
  Horas_de_Estudio: p=0.0000 -> SIGNIFICATIVA


**Interpretacion:**
- **Horas_de_Estudio (p < 0.001):** Contribuye significativamente al rendimiento. Por cada hora
  semanal adicional de estudio, el rendimiento aumenta en promedio ~1.98 puntos.
- **Desayuno Saludable (p = 0.024):** Es significativo, pero con coeficiente negativo (-3.74)
  respecto a la categoria de referencia (Ninguno). Esto puede deberse a confounders en los datos.
- **Desayuno Regular (p = 0.536):** No es estadisticamente significativo.
- **R² = 0.386:** El modelo explica el 38.6% de la variabilidad en el rendimiento.

## 3. Prediccion

In [5]:
nuevo = pd.DataFrame({"Horas_de_Estudio": [5.5], "Desayuno": ["Saludable"]})
pred = res.predict(nuevo)
print(f"Prediccion: estudiante con 5.5 horas de estudio y desayuno Saludable")
print(f"  Rendimiento predicho = {pred.values[0]:.2f} puntos")

Prediccion: estudiante con 5.5 horas de estudio y desayuno Saludable
  Rendimiento predicho = 69.44 puntos


## Conclusiones

- Las variables que contribuyen significativamente al rendimiento son: **Horas_de_Estudio** y
  **Desayuno[Saludable]**. La variable **Desayuno[Regular]** no es significativa.
- El rendimiento predicho para un estudiante que dedica 5.5 horas semanales de estudio y consume
  un desayuno saludable es de aproximadamente **69.44 puntos**.
- La categoria de referencia para Desayuno es "Ninguno". El modelo sugiere que los estudiantes
  que no desayunan tienen mejor rendimiento predicho que los de desayuno saludable, controlando
  por horas de estudio. Este resultado contra-intuitivo podria deberse a variables de confusion
  no incluidas en el modelo.